In [1]:
# --- STEP 1: ENVIRONMENT SETUP AND DEPENDENCY DEPLOYMENT ---
# Install the necessary library suites quietly (-q) inside the execution runtime environment
!pip install -q rank_bm25 nltk

import chromadb
from sentence_transformers import SentenceTransformer
from rank_bm25 import BM25Okapi
import re
from nltk.stem import PorterStemmer

# --- STEP 2: LOCAL VECTOR DATABASE CONNECTION ---
print("⏳ Connecting to ChromaDB...")
# Open a secure connection to the local persistent database storage directory
chroma_client = chromadb.PersistentClient(path="./chroma_database")
# Connect directly to the combined master vector repository collection
collection = chroma_client.get_collection(name="healthcare_knowledge_base")

# --- STEP 3: SEMANTIC ENCODING TRANSFORMS CORE INITIALIZATION ---
print("⏳ Loading BGE-M3 embedding model...")
# Instantiate the embedding model layout across memory architectures for dense matrix vector extraction
embedding_model = SentenceTransformer("BAAI/bge-m3")

# --- STEP 4: LEXICAL COMPRESSION SETUP (PORTER STEMMER) ---
print("⏳ Initializing Porter Stemmer...")
# Instantiate the morphological word stemmer to handle exact match variations
stemmer = PorterStemmer()

def tokenize(text):
    """A fast tokenizer that removes punctuation, lowercases, and stems words for BM25 matching."""
    # Convert incoming strings into completely uniform lowercase text format
    text = text.lower()
    # Apply regular expression borders to extract words, removing punctuation strings natively
    words = re.findall(r'\b\w+\b', text)
    # Strip suffixes down to morphological roots using the Porter stemmer mapping rules
    return [stemmer.stem(w) for w in words]

# --- STEP 5: PAGINATED DATABASE EXTRACTION & LEXICAL CORPUS GENERATION ---
print("⏳ Building the BM25 Keyword Index & extracting scheme vocabulary...")
all_ids = []
tokenized_corpus = []
id_to_category = {}
valid_title_proper_nouns = set()

# Establish memory-safe page chunk definitions for database extraction loops
limit = 2000
offset = 0
all_metadatas = []

# Execute a memory-safe pagination loop across the storage engine to pull data assets safely
while True:
    # Fetch a paginated segment slice specifying limits and shifting offsets to avoid RAM memory spikes
    batch = collection.get(limit=limit, offset=offset, include=['documents', 'metadatas'])
    batch_ids = batch['ids']
    
    # Break out of the loop immediately when the pagination pointer reads an empty payload bucket
    if not batch_ids:
        break
        
    batch_docs = batch['documents']
    batch_metas = batch['metadatas']
    
    # Consolidate paginated batches into unified master lists
    all_ids.extend(batch_ids)
    all_metadatas.extend(batch_metas)
    
    # Process text schemas and tokenize individual items inside current transaction array
    for i, doc_id in enumerate(batch_ids):
        tokenized_corpus.append(tokenize(batch_docs[i]))
        meta = batch_metas[i]
        category = meta.get('category', 'General')
        # Maintain an explicit document-to-category runtime lookup index map
        id_to_category[doc_id] = category
        
    # Increment pagination pointer to pull next index segment slice
    offset += limit

# --- STEP 6: SPARSE RETRIEVAL INDEX INITIALIZATION ---
# Instantiating the optimized BM25 statistical rank tracking engine using the full processed data tokens
bm25_engine = BM25Okapi(tokenized_corpus)

# --- STEP 7: DYNAMIC PROPER NOUN SCHEME VOCABULARY MINING ---
# Establish a strict frequency constraint where lower frequency values represent unique domain proper nouns
IDF_THRESHOLD = 4.0

for meta in all_metadatas:
    # Isolate extraction processes exclusively to state/central government scheme scopes
    if meta.get('category') == "Government Healthcare Scheme":
        title = meta.get('title', '')
        topic = meta.get('topic', '')
        doc_id = meta.get('doc_id', '')
        source_url = meta.get('source_url', '')
        
        # Strategy A: Extract structural proper nouns based on Title Case formatting expressions
        for w in re.findall(r'\b[A-Z][a-zA-Z0-9]*\b', title + " " + topic):
            # Reduce token word entries down to base invariant lexical stems
            stem = stemmer.stem(w.lower())
            # Safely fetch item IDF properties from the engine or assume standard fallback if not visible
            idf = bm25_engine.idf.get(stem, 9.0)
            # Retain only highly-specific title keywords to build custom verification filters
            if idf > IDF_THRESHOLD:
                valid_title_proper_nouns.add(stem)
                
        # Strategy B: Mine hidden program shortcodes/acronyms directly out of metadata schemas
        for text_to_parse in [doc_id, source_url]:
            if text_to_parse:
                # Isolate terminal characters or paths sitting behind separator characters
                last_part = re.split(r'[_/]', text_to_parse)[-1]
                # Strip trailing numerals from slugs to cleanly catch pure alphabetical identifiers
                last_part_clean = re.sub(r'\d+$', '', last_part.lower())
                if last_part_clean:
                    stem = stemmer.stem(last_part_clean)
                    idf = bm25_engine.idf.get(stem, 9.0)
                    if idf > IDF_THRESHOLD:
                        valid_title_proper_nouns.add(stem)

print(f"✅ BM25 Engine successfully indexed {len(tokenized_corpus)} chunks!")

# --- STEP 8: INJECT EXPLICIT FALLBACK ENTRIES ---
# Append localized terms and acronym signatures to prevent routing edge-case failures
valid_title_proper_nouns.update([stemmer.stem(x) for x in ['odisha', 'bsky', 'pmjay', 'jssk', 'jsy', 'pmmvy', 'suman', 'goa', 'rajasthan', 'puducherry', 'pondy', 'pondicherry', 'haryana']])
print(f"✅ Extracted {len(valid_title_proper_nouns)} specific proper nouns for scheme verification.")

⏳ Connecting to ChromaDB...
⏳ Loading BGE-M3 embedding model...


Loading weights:   0%|          | 0/391 [00:00<?, ?it/s]

⏳ Initializing Porter Stemmer...
⏳ Building the BM25 Keyword Index & extracting scheme vocabulary...
✅ BM25 Engine successfully indexed 8461 chunks!
✅ Extracted 738 specific proper nouns for scheme verification.


In [2]:
# --- REVISED CELL 2: Core Retrieval, Auto-Routing & Fusion Engine ---

# Hard distance threshold used to safely reject out-of-bounds queries
DISTANCE_THRESHOLD = 0.39  

def is_unsafe_medical_query(user_query):
    """
    Checks if the user query is asking for prescriptions, dosage, diagnosis, 
    or specific treatment decisions, which violates the chatbot safety guidelines.
    """
    query_clean = user_query.lower()
    
    # Predefined keyword blocklist representing strict medical or prescription boundaries
    unsafe_triggers = [
        "prescribe", "prescription", "dosage", "dose", "mg", "pill", "tablet", 
        "medicine dosage", "treatment decision", "diagnose me", "what medicine should i take",
        "which drug", "drug prescription", "medication choice"
    ]
    
    # Return True if any restricted trigger phrase is found within the processed text
    return any(trigger in query_clean for trigger in unsafe_triggers)

def weighted_reciprocal_rank_fusion(dense_ranks, sparse_ranks, dense_weight=0.85, sparse_weight=0.15, k=60):
    """
    Combines ranks. Dense gets 85% authority to prevent BM25 from burying long documents.
    """
    rrf_scores = {}
    
    # Step A: Compute RRF reciprocal scores for the dense semantic vector search results
    for rank, doc_id in enumerate(dense_ranks):
        if doc_id not in rrf_scores: rrf_scores[doc_id] = 0.0
        rrf_scores[doc_id] += dense_weight * (1.0 / (rank + k))
        
    # Step B: Add RRF reciprocal scores for the sparse BM25 keyword search results
    for rank, doc_id in enumerate(sparse_ranks):
        if doc_id not in rrf_scores: rrf_scores[doc_id] = 0.0
        rrf_scores[doc_id] += sparse_weight * (1.0 / (rank + k))
        
    # Sort the unified chunk collection by their newly aggregated scores in descending order
    sorted_fused_results = sorted(rrf_scores.items(), key=lambda x: x[1], reverse=True)
    return [item[0] for item in sorted_fused_results]

def is_asking_about_scheme(user_query):
    """Identifies if the user's intent is directed toward government welfare programs."""
    query_clean = user_query.lower()
    scheme_keywords = [
        "yojana", "scheme", "kisan", "bima", "pension", "scholarship", 
        "portal", "loan", "subsidy", "benefit", "assistance", "fund", 
        "apply", "disbursed"
    ]
    return any(kw in query_clean for kw in scheme_keywords)

def evaluate_retrieve(user_query, n_results=5):
    # Corrected Proper Noun Guardrail Check
    if is_asking_about_scheme(user_query):
        words = re.findall(r'\b\w+\b', user_query)
        for i, w in enumerate(words):
            # Check capitalized proper nouns (excluding first word of the query)
            if w[0].isupper() and i > 0: 
                stem = stemmer.stem(w.lower())
                stem = re.sub(r'\d+$', '', stem) # Strip trailing numbers
                idf = bm25_engine.idf.get(stem, 9.0)
                
                # If an unrecognized, highly specific proper noun is found, block search immediately
                if idf > 5.5 and len(stem) > 1:
                    if stem not in valid_title_proper_nouns:
                        return [], 1.0
                
    # Helper to split compound multi-part queries (e.g., questions using coordinate clauses)
    def split_query(q):
        splits = re.split(r',\s*and\s+|\s+and\s+(?=does|can|what|how|where|is|are|why|if)', q, flags=re.IGNORECASE)
        splits = [s.strip() for s in splits if s.strip()]
        if len(splits) > 1:
            return splits
        return [q]
        
    sub_queries = split_query(user_query)
    dense_lists = []
    sparse_lists = []
    best_distances = []
    
    # Execute dual search passes for each individual sub-query identified
    for sq in sub_queries:
        # 1. Run Dense Vector Search (Pool size capped at top 50)
        query_vector = embedding_model.encode(sq).tolist()
        chroma_results = collection.query(
            query_embeddings=[query_vector],
            n_results=50, 
            include=['distances']
        )
        if chroma_results['ids'] and chroma_results['ids'][0]:
            dense_lists.append(chroma_results['ids'][0])
            best_distances.append(chroma_results['distances'][0][0])
        else:
            best_distances.append(1.0)
            
        # 2. Run Sparse BM25 Keyword Search (Pool size capped at top 50)
        tokenized_query = tokenize(sq)
        bm25_scores = bm25_engine.get_scores(tokenized_query)
        scored_docs = list(zip(bm25_scores, all_ids))
        scored_docs.sort(key=lambda x: x[0], reverse=True)
        sparse_lists.append([doc_id for score, doc_id in scored_docs][:50])
        
    # Evaluate global guardrail: reject immediately if distance exceeds threshold
    best_semantic_distance = min(best_distances) if best_distances else 1.0
    if best_semantic_distance > DISTANCE_THRESHOLD:
        return [], best_semantic_distance
        
    # Interleave multi-query results using a balanced round-robin strategy
    def interleave_lists(lists):
        interleaved = []
        max_len = max(len(lst) for lst in lists) if lists else 0
        for i in range(max_len):
            for lst in lists:
                if i < len(lst):
                    interleaved.append(lst[i])
        seen = set() # Standard deduplication step
        return [x for x in interleaved if not (x in seen or seen.add(x))]
        
    dense_ids_interleaved = interleave_lists(dense_lists)
    sparse_ids_interleaved = interleave_lists(sparse_lists)
    
    # Run weighted RRF blending step
    final_ranked_ids = weighted_reciprocal_rank_fusion(dense_ids_interleaved, sparse_ids_interleaved)
    
    # Apply document-level diversity filter (max 3 chunks per document source)
    filtered_ranked_ids = []
    doc_counts = {}
    for chunk_id in final_ranked_ids:
        doc_id = chunk_id.split('_chunk_')[0] if '_chunk_' in chunk_id else chunk_id
        current_count = doc_counts.get(doc_id, 0)
        if current_count < 3:
            filtered_ranked_ids.append(chunk_id)
            doc_counts[doc_id] = current_count + 1
            
    # Return final results slice alongside best observed distance metric
    return filtered_ranked_ids[:n_results], best_semantic_distance

def hybrid_retrieve_and_verify(user_query, n_results=3):
    print(f"\n🔍 HYBRID SEARCH (85/15 routed): '{user_query}'")
    
    # Guardrail Pass 1: Handle unsafe clinical queries
    if is_unsafe_medical_query(user_query):
        print("🛑 SAFETY CHECK TRIGGERED: Query violates medical boundaries.")
        print("🤖 Chatbot: 'As an AI health awareness assistant, I cannot provide medical diagnoses, drug prescriptions, medicine dosages, or treatment decisions. Please consult a qualified medical professional for specific clinical advice and treatment.'")
        print("-" * 60)
        return
        
    # Execute retrieval processing pipeline
    retrieved_ids, best_distance = evaluate_retrieve(user_query, n_results=n_results)
    
    # Guardrail Pass 2: Handle out-of-bounds or irrelevant queries
    if not retrieved_ids:
        print(f"🛑 VERIFICATION FAILED: Closest match failed safety check (Distance: {best_distance:.2f}).")
        print("🤖 Chatbot: 'I am sorry, but I do not have enough information in my database to answer your query.'")
        print("-" * 60)
        return
        
    # Output rendering pass for successful verifications
    print(f"✅ VERIFICATION PASSED! (Semantic Distance: {best_distance:.2f})")
    for i, doc_id in enumerate(retrieved_ids):
        doc_data = collection.get(ids=[doc_id], include=['documents', 'metadatas'])
        if doc_data and doc_data['documents']:
            doc_text = doc_data['documents'][0]
            doc_meta = doc_data['metadatas'][0]
            print(f"\n📌 Fused Rank #{i+1} [ID: {doc_id}]")
            print(f"📄 Source: {doc_meta.get('source_name')} ({doc_meta.get('title')})")
            print(f"📝 Text Snippet:\n{doc_text}")
    print("-" * 60)

In [3]:
test_queries = [
    "What are symptoms of diabetes?",
    "Who can apply for PM Kisan?", 
    "When should I see a doctor for dengue?",
    "What causes hypertension?",
    "How is tuberculosis spread?",
]

print("\n🚀 Running Hybrid Search Benchmark...")

for query in test_queries:
    hybrid_retrieve_and_verify(query, n_results=2)


🚀 Running Hybrid Search Benchmark...

🔍 HYBRID SEARCH (85/15 routed): 'What are symptoms of diabetes?'
✅ VERIFICATION PASSED! (Semantic Distance: 0.22)

📌 Fused Rank #1 [ID: who_diabetes_chunk_3]
📄 Source: WHO Fact Sheets (Diabetes)
📝 Text Snippet:
Document: Diabetes
Section: Symptoms

Symptoms of diabetes may occur suddenly. In type 2 diabetes, the symptoms can be mild and may take many years to be noticed. Symptoms of diabetes include: - feeling very thirsty - needing to urinate more often than usual - blurred vision - feeling tired - losing weight unintentionally Over time, diabetes can damage blood vessels in the heart, eyes, kidneys and nerves. People with diabetes have a higher risk of health problems including heart attack, stroke and kidney failure. Diabetes can cause permanent vision loss by damaging blood vessels in the eyes. Many people with diabetes develop problems with their feet from nerve damage and poor blood flow. This can cause foot ulcers and may lead to amputation

In [4]:
# --- NEW CELL 5: Automated RAG Performance Evaluation Layer ---
import json

def run_pipeline_benchmark(test_set_path="golden_test_set.json"):
    print("🚀 Initiating Automated RAG Pipeline Benchmark Evaluation...\n" + "="*85)
    
    # --- STEP 1: LOAD TARGET GOLDEN BENCHMARK DATASET ---
    try:
        with open(test_set_path, 'r', encoding='utf-8') as f:
            evaluation_queries = json.load(f)
    except FileNotFoundError:
        print(f"❌ Error: Could not find '{test_set_path}' in your directory. Please save the JSON file first.")
        return
        
    # --- STEP 2: INITIALIZE PERFORMANCE COUNTERS ---
    total_queries = len(evaluation_queries)
    successful_recalls = 0
    strict_matches = 0
    false_positives = 0
    true_negatives = 0
    mrr_sum = 0.0
    hit_at_1 = 0
    fuzzy_matches = 0
    
    print(f"Loaded {total_queries} evaluation targets. Processing query evaluations:\n" + "-"*85)
    
    # --- STEP 3: EVALUATE EACH QUERY THROUGH THE RETRIEVAL ENGINE ---
    for idx, item in enumerate(evaluation_queries):
        query = item['query']
        expected_ids = item['expected_chunk_ids']
        expected_cat = item['expected_category']
        
        # Execute the full hybrid search and guardrail pipeline to retrieve top 5 results
        retrieved_ids, winning_distance = evaluate_retrieve(query, n_results=5)
        
        print(f"\nTest #{idx+1}: '{query}'")
        print(f"  Category: {expected_cat} | Expected Chunks: {expected_ids}")
        
        # --- SCENARIO A: SAFETY GUARDRAIL EVALUATION (OUT-OF-BOUNDS QUERIES) ---
        if len(expected_ids) == 0:
            if len(retrieved_ids) == 0:
                true_negatives += 1  # Guardrail worked perfectly
                status = "✅ PASSED (Blocked cleanly)"
            else:
                false_positives += 1  # Irrelevant data leaked through
                status = f"❌ FAILED (Guardrail Leak: retrieved {retrieved_ids[0]})"
            print(f"  Status: {status} | Distance Score: {winning_distance:.3f}")
            continue  # Move immediately to the next query
            
        # --- SCENARIO B: VALID CONTENT EVALUATION (RANK & ACCURACY TRACKING) ---
        # Find the rank position of the very first matching chunk in the retrieved list
        first_match_rank = -1
        for rank, rid in enumerate(retrieved_ids):
            if rid in expected_ids:
                first_match_rank = rank
                break
        
        # Calculate Mean Reciprocal Rank (MRR) score contribution
        mrr = 1.0 / (first_match_rank + 1) if first_match_rank != -1 else 0.0
        mrr_sum += mrr
        
        # Calculate Top-1 Hit Rate contribution
        if first_match_rank == 0:
            hit_at_1 += 1
            
        # Calculate Chunk-Level intersection matches
        matched_elements = set(expected_ids).intersection(set(retrieved_ids))
        recall_score = len(matched_elements) / len(expected_ids) if len(expected_ids) > 0 else 0.0
        
        if recall_score > 0:
            successful_recalls += 1  # Pipeline captured at least one true target chunk
            
        if recall_score == 1.0:
            strict_matches += 1  # Pipeline successfully captured 100% of required context
            
        # --- OPTIONAL: CALCULATE DOCUMENT-LEVEL FUZZY MATCHES ---
        # Extract document prefixes by dropping individual '_chunk_' identifiers
        expected_docs = {eid.split('_chunk_')[0] for eid in expected_ids if '_chunk_' in eid}
        retrieved_docs = {rid.split('_chunk_')[0] for rid in retrieved_ids if '_chunk_' in rid}
        doc_matched = len(expected_docs.intersection(retrieved_docs)) > 0
        if doc_matched:
            fuzzy_matches += 1
            
        # --- STEP 4: PRINT INDIVIDUAL QUERY LOGGING REPORTS ---
        if recall_score == 1.0:
            status = "✅ FULL HIT (100% chunks retrieved)"
        elif recall_score > 0.0:
            status = f"⚠️ PARTIAL HIT ({len(matched_elements)}/{len(expected_ids)} chunks)"
        elif doc_matched:
            status = "ℹ️ DOCUMENT MATCH (Wrong chunk, correct document)"
        else:
            status = "❌ MISS"
            
        print(f"  Status: {status} | Distance Score: {winning_distance:.3f}")
        print(f"  Retrieved IDs: {retrieved_ids}")
        
        # Fetch and print small snippets from ChromaDB for rapid manual tracking
        for rid in retrieved_ids[:2]:
            doc_data = collection.get(ids=[rid], include=['documents', 'metadatas'])
            if doc_data and doc_data['documents']:
                text = doc_data['documents'][0].strip().replace('\n', ' ')
                snippet = text[:100] + '...' if len(text) > 100 else text
                source = doc_data['metadatas'][0].get('title', 'Unknown Source')
                print(f"    - [{rid}] (Source: {source}): {snippet}")

    # --- STEP 5: CALCULATE AND AGGREGATE FINAL SYSTEM METRICS ---
    content_queries_count = total_queries - (true_negatives + false_positives)
    guardrail_queries_count = true_negatives + false_positives
    
    # Process ratio splits into percentage rates safely
    recall_rate = (successful_recalls / content_queries_count) * 100 if content_queries_count > 0 else 0
    strict_accuracy = (strict_matches / content_queries_count) * 100 if content_queries_count > 0 else 0
    guardrail_accuracy = (true_negatives / guardrail_queries_count) * 100 if guardrail_queries_count > 0 else 0
    mrr_rate = (mrr_sum / content_queries_count) * 100 if content_queries_count > 0 else 0
    hit_at_1_rate = (hit_at_1 / content_queries_count) * 100 if content_queries_count > 0 else 0
    document_match_rate = (fuzzy_matches / content_queries_count) * 100 if content_queries_count > 0 else 0
    
    # Render final metric visualization panel
    print("\n" + "="*85)
    print("📊 FINAL RETRIEVAL PIPELINE METRIC REPORT:")
    print("="*85)
    print(f"🔹 Strict Chunk Recall@5 Rate : {recall_rate:.2f}%")
    print(f"🔹 Strict Multi-Chunk Accuracy: {strict_accuracy:.2f}%")
    print(f"🔹 Document Recall Rate (Fuzzy): {document_match_rate:.2f}%")
    print(f"🔹 Mean Reciprocal Rank (MRR) : {mrr_rate:.2f}%")
    print(f"🔹 Top-1 Hit Rate (Recall@1)  : {hit_at_1_rate:.2f}%")
    print(f"🔹 Guardrail Rejection Safety : {guardrail_accuracy:.2f}%")
    print("="*85)

# Run the benchmark execution suite
run_pipeline_benchmark()

🚀 Initiating Automated RAG Pipeline Benchmark Evaluation...
Loaded 50 evaluation targets. Processing query evaluations:
-------------------------------------------------------------------------------------

Test #1: 'What is bipolar disorder characterized by?'
  Category: Disease & Awareness | Expected Chunks: ['who_bipolar_disorder_chunk_1']
  Status: ✅ FULL HIT (100% chunks retrieved) | Distance Score: 0.265
  Retrieved IDs: ['who_bipolar_disorder_chunk_1', 'who_bipolar_disorder_chunk_4', 'who_bipolar_disorder_chunk_6', 'who_mental_disorders_chunk_5', 'who_depressive_disorder_depression_chunk_4']
    - [who_bipolar_disorder_chunk_1] (Source: Bipolar disorder): Document: Bipolar disorder Section: Key facts  - Bipolar disorder is a mental health condition that ...
    - [who_bipolar_disorder_chunk_4] (Source: Bipolar disorder): Document: Bipolar disorder Section: Symptoms and patterns  Bipolar disorder is a mental health condi...

Test #2: 'How many people in the world suffer from poor